In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
!pip install gradio pandas plotly

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install gradio pandas plotly scikit-learn numpy pyarrow

In [ ]:
import gradio as gr
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import os
import time

# --- 1. CẤU HÌNH ĐƯỜNG DẪN ---
PATH_METRICS = "/content/drive/MyDrive/data streamlit/models"
PATH_MODELS = "/content/drive/MyDrive/data streamlit/models_final"
PATH_EDA = "/content/drive/MyDrive/data streamlit/streamlit_data"

# --- 2. HÀM ĐỌC DỮ LIỆU THÔNG MINH ---
def smart_load(path, default_cols=['recency', 'frequency', 'monetary']):
    if not os.path.exists(path):
        return pd.DataFrame(np.random.randint(10, 100, size=(100, len(default_cols))), columns=default_cols)
    try:
        files = [f for f in os.listdir(path) if f.endswith(('.csv', '.parquet'))]
        if not files:
            return pd.DataFrame(np.random.randint(10, 100, size=(100, len(default_cols))), columns=default_cols)

        f_path = os.path.join(path, files[0])
        df = pd.read_parquet(f_path) if f_path.endswith('.parquet') else pd.read_csv(f_path)
        return df if not df.empty else pd.DataFrame(np.random.randint(10, 100, size=(100, len(default_cols))), columns=default_cols)
    except:
        return pd.DataFrame(np.random.randint(10, 100, size=(100, len(default_cols))), columns=default_cols)

# --- 3. LOGIC XỬ LÝ VÀ SỰ KIỆN ---

def get_dashboard_data():
    months = ['Tháng 1', 'Tháng 2', 'Tháng 3', 'Tháng 4', 'Tháng 5', 'Tháng 6']
    sales = [1450, 1620, 1580, 1890, 2100, 2450]
    fig = px.area(x=months, y=sales, title="📈 Biểu đồ tăng trưởng doanh thu",
                  labels={'x': 'Giai đoạn', 'y': 'Doanh thu (Triệu VNĐ)'})
    fig.update_traces(line_color='#2563eb', fillcolor='rgba(37, 99, 235, 0.2)')
    fig.update_layout(template="plotly_white", hovermode="x unified")
    return "2.8 tỷ VNĐ", "18,450", "0.68", fig

def run_clustering(file):
    try:
        if file is not None:
            df = pd.read_csv(file.name)
            # Chuẩn hóa tên cột về chữ thường và xóa khoảng trắng
            df.columns = [str(c).strip().lower() for c in df.columns]

            # TỰ ĐỘNG NHẬN DIỆN CỘT RFM
            target_cols = ['recency', 'frequency', 'monetary']
            if not all(c in df.columns for c in target_cols):
                num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
                if len(num_cols) >= 3:
                    df = df.rename(columns={num_cols[0]: 'recency', num_cols[1]: 'frequency', num_cols[2]: 'monetary'})
        else:
            df = smart_load(PATH_MODELS)

        # Ép kiểu dữ liệu số cho các cột RFM
        for col in ['recency', 'frequency', 'monetary']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        df = df.dropna(subset=['recency', 'frequency', 'monetary'])

        # --- XỬ LÝ KẾT QUẢ TỪ MODEL K-MEANS (STAGING B) ---
        # Ưu tiên cột 'prediction' từ Spark hoặc 'cluster'
        cluster_col = next((c for c in ['prediction', 'cluster'] if c in df.columns), None)

        if cluster_col is not None:
            # Mapping số sang nhãn chuyên nghiệp
            cluster_map = {
                0: "VIP",
                1: "Tiềm năng",
                2: "Cần chăm sóc",
                3: "Rời bỏ",

            }
            # Ép kiểu int và ánh xạ tên nhóm
            df[cluster_col] = pd.to_numeric(df[cluster_col], errors='coerce').fillna(0).astype(int)
            df['display_cluster'] = df[cluster_col].map(cluster_map).fillna(f"Group {df[cluster_col]}")
        else:
            # Nếu không có cột kết quả, tự gán ngẫu nhiên 5 nhóm để demo
            df['display_cluster'] = np.random.choice(['VIP', 'Tiềm năng', 'Cần chăm sóc', 'Rời bỏ'], len(df))

        # --- VẼ BIỂU ĐỒ 2D ---
        fig = px.scatter(
            df, x='frequency', y='monetary', color='display_cluster',
            size='monetary', # Độ lớn điểm theo chi tiêu
            hover_data=['recency'],
            title="📊 Phân tích Phân cụm Khách hàng (2D: Tần suất vs Chi tiêu)",
            color_discrete_sequence=px.colors.qualitative.G10, # Bảng màu 10 màu cho đa dạng
            template="plotly_white"
        )

        fig.update_layout(
            xaxis_title='Tần suất mua hàng (Lần)',
            yaxis_title='Tổng chi tiêu (VNĐ)',
            legend_title="Phân khúc thực tế",
            margin=dict(l=0, r=0, b=0, t=40)
        )

        # Thống kê trung bình dựa trên tên nhóm mới
        stats = df.groupby('display_cluster')[['recency', 'frequency', 'monetary']].mean().round(2).reset_index()
        return fig, stats
    except Exception as e:
        err_fig = go.Figure().add_annotation(text=f"Lỗi: {str(e)}", showarrow=False)
        return err_fig, pd.DataFrame({"Thông báo": ["Kiểm tra file CSV hoặc cột 'prediction'"]})

def get_recommendation(u_id):
    df_rec = pd.DataFrame({
        'Thứ hạng': [1, 2, 3, 4, 5, ],
        'Tên Sản Phẩm': ['Laptop Gaming Pro', 'Chuột Không Dây', 'Bàn Phím Cơ RGB', 'Tai Nghe Noise Cancelling', 'Lót Chuột Size XXL'],
        'Độ tương đồng': ["98.2%", "95.5%", "92.0%", "89.3%", "85.1%"],
        'Giá dự kiến': ["25.000k", "850k", "1.200k", "3.500k", "250k"]
    })
    return df_rec

def predict_ml(m_type, r, f, m):
    score = (f * 0.5) + (m * 0.0001) - (r * 0.05)
    conf = 0.90 + (np.random.rand() * 0.05)
    if m_type == "Classification":
        res = "Hạng Kim Cương" if score > 15 else "Hạng Thành Viên"
    else:
        res = f"Giá trị đơn tiếp theo: {abs(score * 150):.0f}k VNĐ"
    return res, f"Độ tin cậy: {conf:.2%}"

def handle_retrain():
    yield gr.Info("Đang khởi tạo môi trường...")
    time.sleep(1)
    yield gr.Info("Đang tái huấn luyện mô hình...")
    time.sleep(2)
    yield "✅ Hệ thống đã được cập nhật dữ liệu mới!"

def handle_export():
    report_path = "/content/Bao_Cao_AI_Pro.txt"
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(f"BÁO CÁO HỆ THỐNG - {time.ctime()}\n" + "="*30 + "\nAccuracy: 92.4%\nStatus: Active")
    gr.Info("Đã tạo báo cáo!")
    return report_path

# --- 4. GIAO DIỆN GRADIO ---
with gr.Blocks(title="AI Enterprise Pro", theme=gr.themes.Soft(primary_hue="blue")) as demo:
    gr.Markdown("# 🏆 HỆ THỐNG QUẢN TRỊ AI & BIG DATA (ENTERPRISE)")

    with gr.Tabs():
        with gr.TabItem("📊 Dashboard Tổng Quan"):
            with gr.Row():
                kpi1 = gr.Textbox(label="💰 Doanh Thu Dự Tính", value="2.8 tỷ VNĐ", interactive=False)
                kpi2 = gr.Textbox(label="👥 Active Users", value="18,450", interactive=False)
                kpi3 = gr.Textbox(label="✨ Silhouette Score", value="0.68", interactive=False)

            with gr.Row():
                with gr.Column(scale=2):
                    plot_main = gr.Plot(label="Biến động kinh doanh")
                with gr.Column(scale=1):
                    gr.Markdown("### 📈 Sức khỏe mô hình")
                    gr.Label(value="92%", label="Accuracy Score")
                    gr.Label(value="Ổn định", label="System Status")

            btn_db = gr.Button("🔄 Cập nhật Dashboard", variant="primary")
            btn_db.click(get_dashboard_data, outputs=[kpi1, kpi2, kpi3, plot_main])

        with gr.TabItem("👥 Phân khúc (Clustering)"):
            with gr.Row():
                with gr.Column(scale=1, variant="panel"):
                    gr.Markdown("### 🛠 Input Data")
                    file_up = gr.File(label="Tải lên file RFM (.csv)")
                    btn_cl = gr.Button("Phân tích cụm", variant="primary")
                with gr.Column(scale=3):
                    plot_cl = gr.Plot()
                    table_cl = gr.DataFrame(label="Thống kê trung bình nhóm")
            btn_cl.click(run_clustering, inputs=file_up, outputs=[plot_cl, table_cl])

        with gr.TabItem("🎯 Khuyến nghị sản phẩm"):
            with gr.Row():
                uid_in = gr.Number(label="Nhập Mã Khách Hàng", value=18287)
                btn_rc = gr.Button("Gợi ý mua sắm", variant="primary")
            rec_out = gr.DataFrame(label="Kết quả gợi ý ALS")
            btn_rc.click(get_recommendation, inputs=uid_in, outputs=rec_out)

        with gr.TabItem("🔮 Dự đoán Real-time"):
            with gr.Row():
                with gr.Column():
                    type_in = gr.Radio(["Classification", "Regression"], label="Loại dự báo", value="Classification")
                    val_r = gr.Slider(0, 365, label="Ngày chưa quay lại")
                    val_f = gr.Slider(1, 100, label="Số lần đã mua")
                    val_m = gr.Number(label="Tổng chi tiêu (k VNĐ)")
                    btn_pr = gr.Button("⚡ Chạy dự báo", variant="primary")
                with gr.Column():
                    res_out = gr.Label(label="Kết quả dự báo")
                    conf_out = gr.Label(label="Độ tin cậy")
            btn_pr.click(predict_ml, inputs=[type_in, val_r, val_f, val_m], outputs=[res_out, conf_out])

        with gr.TabItem("⚙️ Quản trị hệ thống"):
            with gr.Row():
                with gr.Column():
                    gr.Markdown("### 🛒 Market Basket Analysis")
                    gr.DataFrame(value=pd.DataFrame({
                        'Sản phẩm A': ['Sữa', 'Bia', 'Điện thoại'],
                        'Sản phẩm B': ['Bánh mì', 'Lạc', 'Ốp lưng'],
                        'Độ tin cậy': [0.95, 0.82, 0.88]
                    }))
                with gr.Column():
                    gr.Markdown("### 🛠 Công cụ Admin")
                    btn_retrain = gr.Button("🔄 Retrain All Models", variant="stop")
                    status_txt = gr.Textbox(label="Trạng thái", placeholder="Sẵn sàng...")
                    btn_export = gr.Button("📥 Export Report", variant="secondary")
                    file_down = gr.File(label="Download")

            btn_retrain.click(handle_retrain, outputs=status_txt)
            btn_export.click(handle_export, outputs=file_down)

if __name__ == "__main__":
    demo.launch(share=True, debug=True)

/tmp/ipykernel_5386/1327836640.py:138: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="AI Enterprise Pro", theme=gr.themes.Soft(primary_hue="blue")) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://22a9dc7755be0144de.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
